<a href="https://colab.research.google.com/github/Imran0324/Ml-Internship-Assignment/blob/main/notebooks/02_your_first_readable_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Imran0324/Ml-Internship-Assignment/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [10]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.columns.tolist())

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [2]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,61678,194,19.7,0.15,down
16514,59472,194,24.8,0.13,down
7021,25715,194,22.2,0.23,down
21268,13299,193,10.5,0.49,down
11489,7812,194,39.0,0.01,down
12045,7558,193,17.9,0.20,down
698,4590,194,31.0,0.00,down
5327,4556,194,16.4,0.33,down
26810,4429,194,25.3,0.38,down
20837,1697,193,15.8,0.12,down


We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.680


## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [11]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
features_new = [
    "content_age_days",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "word_count",
    "engagement_rate"
]

X_new = df[features_new].replace(
    [np.inf, -np.inf], np.nan
).fillna(0)

tree_new = DecisionTreeClassifier(
    max_depth=2,
    class_weight="balanced",
    random_state=42
)

tree_new.fit(X_new, y)

print("New feature tree:")
print(export_text(tree_new, feature_names=features_new))

new_scores = tree_new.predict_proba(X_new)[:, 1]

print("Precision@50 =", precision_at_k(new_scores, y, 50))

X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

New feature tree:
|--- avg_position <= 0.55
|   |--- avg_position <= 0.15
|   |   |--- class: 0
|   |--- avg_position >  0.15
|   |   |--- class: 0
|--- avg_position >  0.55
|   |--- content_age_days <= 287.50
|   |   |--- class: 1
|   |--- content_age_days >  287.50
|   |   |--- class: 0

Precision@50 = 0.7
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [5]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.900   vs   tree 0.550
Precision@50:  hand rule 0.680   vs   tree 0.600


Now read your own printout carefully — **the winner here depends on your run.** A depth-2 tree can only give four different scores (one per leaf), so the "top 50" is mostly one big block of tied pages, and different library versions break those ties differently. On some stacks the tree wins at Precision@50; on others the hand rule holds both. **Both results are real.** The stable lesson: a sharp human rule can be excellent at the very top of the list; a model's advantage — when it shows up — appears deeper, where simple rules run out of signal; and any comparison built on heavily tied scores is fragile. Saying exactly what YOUR run shows — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [6]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

In [9]:
# Your experiment here

# Test different tree depths

for depth in [3, 4]:
    test_tree = DecisionTreeClassifier(
        max_depth=depth,
        class_weight="balanced",
        random_state=42
    )

    test_tree.fit(X, y)

    scores = test_tree.predict_proba(X)[:, 1]

    print("max_depth =", depth)
    print("Precision@50 =", precision_at_k(scores, y, 50))
    print()
 ### My Experiment

""" I tested different decision tree depths.

With `max_depth=3`, Precision@50 was **0.72**.
With `max_depth=4`, Precision@50 was **0.68**.

The `max_depth=3` tree performed better than the `max_depth=4` tree in my experiment.
Therefore, increasing the depth from 3 to 4 did not improve Precision@50.

I then removed `impressions_90d` and added `engagement_rate`.
The tree chose **`avg_position`** as its first split.

The new feature set produced a Precision@50 of **0.70**.

# Final Experiment: Client-Holdout Validation
# We split the 32 unique clients into 25 training clients and 7 unseen test clients.
# This ensures that the model is evaluated on clients it has never seen during training.

# Use the features selected in the new feature experiment.
# avg_position was the first and most important split in the new decision tree.
features = [
    "avg_position",
    "content_age_days",
    "engagement_rate"
]

# Create training and testing datasets based on client IDs.
# Training data contains 25 clients, while testing data contains 7 unseen clients.

# Train a Decision Tree with max_depth=3.
# This depth performed better than max_depth=4 in the previous experiment.

# Predict the probability that each test page is declining.
# The test set contains 3,419 rows from the 7 unseen clients.

# Rank the test pages by their predicted probability of declining.
# Precision@50 measures how many of the top 50 predictions are actually declining.

# Result:
# Precision@50 = 0.60
# 30 out of the top 50 predicted declining pages were actually declining.
# This shows how well the model generalizes to completely unseen clients.



"""





max_depth = 3
Precision@50 = 0.72

max_depth = 4
Precision@50 = 0.68



In [12]:
print(df["client_id"].nunique())
print(df["client_id"].head())

32
0    client_f369cb89fc
1    client_4e07408562
2    client_7f2253d7e2
3    client_19581e27de
4    client_3fdba35f04
Name: client_id, dtype: object


In [13]:
from sklearn.model_selection import train_test_split

# Get unique client IDs
clients = df["client_id"].unique()

# Split clients: 80% for training, 20% for testing
train_clients, test_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=42
)

print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))

Training clients: 25
Testing clients: 7


In [14]:
# Create masks for training and testing clients
train_mask = df["client_id"].isin(train_clients)
test_mask = df["client_id"].isin(test_clients)

# Use the same features from your new experiment
features = [
    "avg_position",
    "content_age_days",
    "engagement_rate"
]

# Training data
X_train = df.loc[train_mask, features]
y_train = df.loc[train_mask, "target"]

# Testing data
X_test = df.loc[test_mask, features]
y_test = df.loc[test_mask, "target"]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

KeyError: 'target'

In [15]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label']


In [16]:
# Features from our new experiment
features = [
    "avg_position",
    "content_age_days",
    "engagement_rate"
]

# Create masks for training and testing clients
train_mask = df["client_id"].isin(train_clients)
test_mask = df["client_id"].isin(test_clients)

# Training data
X_train = df.loc[train_mask, features]
y_train = df.loc[train_mask, "is_declining_label"]

# Testing data
X_test = df.loc[test_mask, features]
y_test = df.loc[test_mask, "is_declining_label"]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 26581
Testing rows: 3419


In [17]:
from sklearn.tree import DecisionTreeClassifier

# Create decision tree
model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

# Train ONLY on the training clients
model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [18]:
# Predict the probability of being "declining"
y_test_scores = model.predict_proba(X_test)[:, 1]

print("Predictions created:", len(y_test_scores))

Predictions created: 3419


In [19]:
# Create a results table
results = df.loc[test_mask, ["client_id"]].copy()

results["actual"] = y_test.values
results["score"] = y_test_scores

# Sort by the model's predicted probability
results = results.sort_values("score", ascending=False)

# Take the top 50 predictions
top_50 = results.head(50)

# Calculate Precision@50
precision_at_50 = top_50["actual"].mean()

print("Precision@50:", precision_at_50)
print("Declining pages in Top 50:", top_50["actual"].sum())

Precision@50: 0.6
Declining pages in Top 50: 30


### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.